# Chapter 4: Playwright and Dynamic Content

Client-side rendered (CSR) sites are invisible to `BeautifulSoup`. The server sends an empty HTML shell,
and JavaScript running in the browser fetches data and builds the DOM after page load.

**Playwright** is a headless browser automation library that actually executes JavaScript, giving you
the rendered DOM just as a user would see it.

This notebook:
1. Proves the CSR problem concretely
2. Shows Playwright as the solution
3. Demonstrates auto-detection of CSR vs SSR
4. Covers wait strategies for dynamic content
5. Shows the probe-then-escalate pattern

In [9]:
import httpx
import asyncio
from bs4 import BeautifulSoup

SSR_URL = "http://localhost:8001/products"
CSR_URL = "http://localhost:8002/products"

## 4.1 The Problem: Static Fetch on a CSR Site

In [10]:
# Static fetch both sites
ssr_html = httpx.get(SSR_URL).text
csr_html = httpx.get(CSR_URL).text

ssr_soup = BeautifulSoup(ssr_html, "lxml")
csr_soup = BeautifulSoup(csr_html, "lxml")

# What does BeautifulSoup find?
print("=== SSR Site ===")
ssr_cards = ssr_soup.select(".product-card")
print(f"Product cards: {len(ssr_cards)}")
print(f"First product: {ssr_cards[0].select_one('h2').get_text(strip=True) if ssr_cards else 'N/A'}")

print()
print("=== CSR Site ===")
csr_cards = csr_soup.select(".product-card")
print(f"Product cards: {len(csr_cards)}")
visible_text = csr_soup.get_text(strip=True)
print(f"Visible text ({len(visible_text)} chars): {visible_text[:150]!r}")

=== SSR Site ===
Product cards: 10
First product: MacBook Pro 14"

=== CSR Site ===
Product cards: 0
Visible text (159 chars): 'ShopSphereShopSphereAll ProductsCSRLoading productsShopSphere CSR Demo - Content rendered client-side via JavaScript. View source to confirm the HTML '


## 4.2 Detecting CSR vs SSR

Before scraping, probe the page to detect if JavaScript rendering is needed.

In [11]:
def detect_render_mode(url: str, probe_selector: str = None) -> str:
    """
    Probe a URL to determine if static scraping is sufficient.
    Returns 'static' or 'playwright'.
    """
    resp = httpx.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
    html = resp.text
    soup = BeautifulSoup(html, "lxml")
    
    # Signal 1: very little visible text
    text = soup.get_text(strip=True)
    if len(text) < 200:
        return "playwright"  # CSR: empty shell
    
    # Signal 2: probe selector returns nothing
    if probe_selector and not soup.select(probe_selector):
        return "playwright"  # Content selector matches nothing
    
    # Signal 3: page is a SPA (has module scripts, no real content)
    scripts = soup.select('script[type="module"], script[src*="chunk"], script[src*="bundle"]')
    if len(scripts) > 2 and len(text) < 500:
        return "playwright"
    
    return "static"

print(f"SSR site render mode: {detect_render_mode(SSR_URL, '.product-card')}")
print(f"CSR site render mode: {detect_render_mode(CSR_URL, '.product-card')}")
print(f"JobHive SSR render mode: {detect_render_mode('http://localhost:8003', '.job-card')}")
print(f"JobHive CSR render mode: {detect_render_mode('http://localhost:8004', '.job-card')}")

SSR site render mode: static
CSR site render mode: playwright
JobHive SSR render mode: static
JobHive CSR render mode: playwright


## 4.3 Playwright: Fetching Rendered HTML

In [12]:
# Install playwright if needed:
# pip install playwright && playwright install chromium

from playwright.async_api import async_playwright

async def fetch_with_playwright(url: str, wait_for_selector: str = None) -> str:
    """
    Fetch a URL using a headless Chromium browser.
    Returns the full rendered HTML after JavaScript executes.
    """
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        
        # Wait for network to be idle (JS fetches complete)
        await page.goto(url, wait_until="networkidle", timeout=30000)
        
        # Optionally wait for a specific element to appear
        if wait_for_selector:
            try:
                await page.wait_for_selector(wait_for_selector, timeout=10000)
            except Exception:
                pass  # Continue even if selector never appears
        
        html = await page.content()
        await browser.close()
    return html

# Fetch the CSR site with Playwright
csr_rendered = await fetch_with_playwright(CSR_URL, wait_for_selector=".product-card")

rendered_soup = BeautifulSoup(csr_rendered, "lxml")
cards = rendered_soup.select(".product-card")
print(f"After Playwright: {len(cards)} product cards found")
if cards:
    print(f"First product: {cards[0].select_one('h2').get_text(strip=True)}")
    print(f"Price: {cards[0].select_one('.product-price').get_text(strip=True)}")

After Playwright: 10 product cards found
First product: MacBook Pro 14"
Price: $1999.00


## 4.4 The Auto-Fallback Pattern

Playwright is slower and heavier than `httpx`. The optimal strategy is:
1. Try static fetch first
2. Check if expected content appeared
3. Fall back to Playwright only if static fetch failed

This is the core of the bylgja `render_mode=auto` implementation.

In [13]:
import time

async def fetch_with_fallback(
    url: str,
    probe_selector: str,
    verbose: bool = True,
) -> tuple[str, str]:  # (html, render_mode)
    """
    Try static fetch first. If the probe selector returns nothing,
    escalate to Playwright. Returns (html, render_mode_used).
    """
    t0 = time.time()
    
    # Step 1: static fetch
    try:
        static_html = httpx.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10).text
        soup = BeautifulSoup(static_html, "lxml")
        
        if soup.select(probe_selector):
            if verbose:
                print(f"Static fetch succeeded in {time.time()-t0:.2f}s")
            return static_html, "static"
    except Exception as e:
        if verbose:
            print(f"Static fetch failed: {e}")
    
    # Step 2: escalate to Playwright
    if verbose:
        print(f"Probe selector '{probe_selector}' matched nothing - escalating to Playwright")
    
    t1 = time.time()
    playwright_html = await fetch_with_playwright(url, wait_for_selector=probe_selector)
    if verbose:
        print(f"Playwright fetch succeeded in {time.time()-t1:.2f}s")
    return playwright_html, "playwright"


# SSR site - static succeeds
print("=== SSR Site ===")
html, mode = await fetch_with_fallback(SSR_URL, ".product-card")
print(f"Render mode used: {mode}")
print()

# CSR site - static fails, Playwright kicks in
print("=== CSR Site ===")
html, mode = await fetch_with_fallback(CSR_URL, ".product-card")
soup = BeautifulSoup(html, "lxml")
print(f"Render mode used: {mode}")
print(f"Products found: {len(soup.select('.product-card'))}")

=== SSR Site ===
Static fetch succeeded in 0.05s
Render mode used: static

=== CSR Site ===
Probe selector '.product-card' matched nothing - escalating to Playwright
Playwright fetch succeeded in 2.27s
Render mode used: playwright
Products found: 10


## 4.5 Playwright Wait Strategies

JavaScript apps load at different speeds. You need the right wait strategy for each.

In [14]:
async def compare_wait_strategies(url: str):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        
        strategies = [
            ("load", "DOMContentLoaded + load event"),
            ("domcontentloaded", "DOMContentLoaded only (fast)"),
            ("networkidle", "Wait for no network requests for 500ms (recommended)"),
        ]
        
        for strategy, description in strategies:
            t0 = time.time()
            page = await browser.new_page()
            await page.goto(url, wait_until=strategy, timeout=15000)
            html = await page.content()
            soup = BeautifulSoup(html, "lxml")
            cards = soup.select(".product-card, .job-card")
            elapsed = time.time() - t0
            print(f"{strategy:20} | {elapsed:.2f}s | {len(cards):>3} items | {description}")
            await page.close()
        
        await browser.close()

print("Wait strategies on CSR ShopSphere:")
print(f"{'Strategy':20} | {'Time':>5} | {'Items':>5} | Description")
print("-" * 70)
await compare_wait_strategies(CSR_URL)

Wait strategies on CSR ShopSphere:
Strategy             |  Time | Items | Description
----------------------------------------------------------------------
load                 | 0.13s |  10 items | DOMContentLoaded + load event
domcontentloaded     | 0.09s |  10 items | DOMContentLoaded only (fast)
networkidle          | 1.57s |  10 items | Wait for no network requests for 500ms (recommended)


## 4.6 Infinite Scroll

Some sites load more content as you scroll down rather than paginating. Playwright can simulate scrolling.

In [15]:
async def scrape_infinite_scroll(
    url: str,
    item_selector: str,
    max_scrolls: int = 5,
) -> list:
    """
    Scroll to bottom repeatedly to trigger content loading.
    Collect items after each scroll.
    """
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(url, wait_until="networkidle")
        
        all_items = set()
        prev_count = 0
        
        for scroll_num in range(max_scrolls):
            # Scroll to bottom
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            
            # Wait for new content
            await page.wait_for_timeout(1500)
            
            html = await page.content()
            soup = BeautifulSoup(html, "lxml")
            items = soup.select(item_selector)
            current_count = len(items)
            
            print(f"Scroll {scroll_num + 1}: {current_count} items (new: {current_count - prev_count})")
            
            if current_count == prev_count:
                print("No new items after scroll - end of content")
                break
            
            prev_count = current_count
            all_items = items  # Keep latest snapshot
        
        await browser.close()
        return all_items

# Our demo sites paginate, not infinite scroll, but the pattern is the same
# This function works against any infinite-scroll site
print("Note: demo sites use pagination, not infinite scroll.")
print("The scrape_infinite_scroll() function above works against real infinite-scroll sites.")
print("Example usage:")
print("  items = await scrape_infinite_scroll('https://example.com/feed', '.post-card')")

Note: demo sites use pagination, not infinite scroll.
The scrape_infinite_scroll() function above works against real infinite-scroll sites.
Example usage:
  items = await scrape_infinite_scroll('https://example.com/feed', '.post-card')


## 4.7 Key Takeaways

1. **CSR sites are invisible to BeautifulSoup** - the HTML is an empty shell
2. **Playwright executes JavaScript** and gives you the rendered DOM
3. **Auto-detect rendering mode**: probe with a content selector; if it matches nothing, escalate
4. **networkidle** is the most reliable wait strategy for API-driven SPAs
5. **Playwright is slower** - use it as a fallback, not the default
6. **The bylgja config `render_mode`** encodes this decision: `static`, `playwright`, or `auto`